# DAX Helpers

> This applies to Analytics_DataModel_v3.

These scripts help build the different measures and tables based on the metrics available in the database.

Run the notebook to generate the DAX/TMDL used to add the tables/measures to the data model.

In [ ]:
# Set up the list of columns. Key is the group name and the value is the list of columns in that group
# This list must be kept in sync with the columns in the profiling.ActivitiesWeeklyColumns table
# @ src\AnalyticsEngine\App.ControlPanel.Engine\SqlExtentions\Profiling-03-CreateSchema.sql
activity_columns = {
    "OneDrive": [
        "OneDrive Viewed/Edited", "OneDrive Synced", "OneDrive Shared Internally", "OneDrive Shared Externally"
    ],
    "Outlook": [
        "Emails Sent", "Emails Received", "Emails Read", "Outlook Meetings Created", "Outlook Meetings Interacted"
        ],
    "SharePoint Online": [
        "SPO Viewed/Edited", "SPO Synced", "SPO Shared Internally", "SPO Shared Externally"
        ],
    "Teams": [
        "Teams Private Chats", "Teams Team Chats", "Teams Calls",
        "Teams Meetings", "Teams Meetings Attended", "Teams Meetings Organized",
        "Teams Adhoc Meetings Attended", "Teams Adhoc Meetings Organized",
        "Teams Scheduled Onetime Meetings Attended", "Teams Scheduled Onetime Meetings Organized",
        "Teams Scheduled Recurring Meetings Attended", "Teams Scheduled Recurring Meetings Organized",
        "Teams Post Messages", "Teams Reply Messages", "Teams Urgent Messages"
        ],
    "Teams Durations": [
        "Teams Audio Duration Seconds", "Teams Video Duration Seconds", "Teams Screenshare Duration Seconds"
    ],
    "Yammer": [
        "Yammer Posted", "Yammer Read", "Yammer Liked"
    ],
    "Copilot": [
        "Copilot Chats", "Copilot Meetings", "Copilot Files"
    ],
    "Copilot Apps": [
        "Copilot App Assist365", "Copilot App Bing", "Copilot App BashTool", "Copilot App DevUI", "Copilot App Excel",
        "Copilot App Loop", "Copilot App M365AdminCenter", "Copilot App M365App", "Copilot App Office", "Copilot App OneNote",
        "Copilot App Outlook", "Copilot App Planner", "Copilot App PowerPoint", "Copilot App SharePoint", "Copilot App Stream",
        "Copilot App Teams", "Copilot App VivaCopilot", "Copilot App VivaEngage", "Copilot App VivaGoals", "Copilot App Whiteboard",
        "Copilot App Word"
    ]
}

## TMDL to create the activity measures

For each new metric that needs a measure to be created in the _Measures table:

```TMDL
createOrReplace
    table _Measures
        measure 'Copilot App Loop' = SUM('Weekly Activities'[Copilot App Loop])
            formatString: #,0
            displayFolder: Activities\Copilot Apps
```

In [6]:
# Print the TMDL to add measures to the _Measures table
template = """        measure '$name' = SUM('Weekly Activities'[$name])
            formatString: #,0
            displayFolder: Activities\$group"""
for group, column_list in activity_columns.items():
    for name in column_list:
        print(template.replace("$name", name).replace("$group", group))

        measure 'OneDrive Viewed/Edited' = SUM('Weekly Activities'[OneDrive Viewed/Edited])
            formatString: #,0
            displayFolder: Activities\OneDrive
        measure 'OneDrive Synced' = SUM('Weekly Activities'[OneDrive Synced])
            formatString: #,0
            displayFolder: Activities\OneDrive
        measure 'OneDrive Shared Internally' = SUM('Weekly Activities'[OneDrive Shared Internally])
            formatString: #,0
            displayFolder: Activities\OneDrive
        measure 'OneDrive Shared Externally' = SUM('Weekly Activities'[OneDrive Shared Externally])
            formatString: #,0
            displayFolder: Activities\OneDrive
        measure 'Emails Sent' = SUM('Weekly Activities'[Emails Sent])
            formatString: #,0
            displayFolder: Activities\Outlook
        measure 'Emails Received' = SUM('Weekly Activities'[Emails Received])
            formatString: #,0
            displayFolder: Activities\Outlook
        measure 'Email

## Activities table

This table contains a column with all the metrics names and one measure with a switch that will render the selected metric:

In [7]:
# Print the DAX to create the Activity metrics table
table = """Activity metrics = 
DATATABLE(
    "Metric", STRING,
    {
$values
    }
)
"""
template = """    {"$name"}"""
values = []
for group, column_list in activity_columns.items():
    for name in column_list:
        values.append(template.replace("$name", name))

print(table.replace("$values", ",\n".join(values)))

Activity metrics = 
DATATABLE(
    "Metric", STRING,
    {
    {"OneDrive Viewed/Edited"},
    {"OneDrive Synced"},
    {"OneDrive Shared Internally"},
    {"OneDrive Shared Externally"},
    {"Emails Sent"},
    {"Emails Received"},
    {"Emails Read"},
    {"Outlook Meetings Created"},
    {"Outlook Meetings Interacted"},
    {"SPO Viewed/Edited"},
    {"SPO Synced"},
    {"SPO Shared Internally"},
    {"SPO Shared Externally"},
    {"Teams Private Chats"},
    {"Teams Team Chats"},
    {"Teams Calls"},
    {"Teams Meetings"},
    {"Teams Meetings Attended"},
    {"Teams Meetings Organized"},
    {"Teams Adhoc Meetings Attended"},
    {"Teams Adhoc Meetings Organized"},
    {"Teams Scheduled Onetime Meetings Attended"},
    {"Teams Scheduled Onetime Meetings Organized"},
    {"Teams Scheduled Recurring Meetings Attended"},
    {"Teams Scheduled Recurring Meetings Organized"},
    {"Teams Audio Duration Seconds"},
    {"Teams Video Duration Seconds"},
    {"Teams Screenshare Duration 

In [8]:
# Print the DAX to create the Activity value measure
measure = """Activity value = 
SWITCH(
    SELECTEDVALUE('Activity metrics'[Name]),
$values
)
"""
template = """    "$name", [$name]"""
values = []
for group, column_list in activity_columns.items():
    for name in column_list:
        values.append(template.replace("$name", name))

print(measure.replace("$values", ",\n".join(values)))

Activity value = 
SWITCH(
    SELECTEDVALUE('Activity metrics'[Name]),
    "OneDrive Viewed/Edited", [OneDrive Viewed/Edited],
    "OneDrive Synced", [OneDrive Synced],
    "OneDrive Shared Internally", [OneDrive Shared Internally],
    "OneDrive Shared Externally", [OneDrive Shared Externally],
    "Emails Sent", [Emails Sent],
    "Emails Received", [Emails Received],
    "Emails Read", [Emails Read],
    "Outlook Meetings Created", [Outlook Meetings Created],
    "Outlook Meetings Interacted", [Outlook Meetings Interacted],
    "SPO Viewed/Edited", [SPO Viewed/Edited],
    "SPO Synced", [SPO Synced],
    "SPO Shared Internally", [SPO Shared Internally],
    "SPO Shared Externally", [SPO Shared Externally],
    "Teams Private Chats", [Teams Private Chats],
    "Teams Team Chats", [Teams Team Chats],
    "Teams Calls", [Teams Calls],
    "Teams Meetings", [Teams Meetings],
    "Teams Meetings Attended", [Teams Meetings Attended],
    "Teams Meetings Organized", [Teams Meetings Organiz

## List of usage columns in the database

Most columns are bit/boolean except for "Yammer Platform Count" that is a smallint.
They indicate if an app/platform combination was used that week.

In [1]:
usage_columns = {
    "Teams": [
        "Teams Used Web", "Teams Used Mobile"
        "Teams Used Mac", "Teams Used Windows", "Teams Used Linux", "Teams Used Chrome OS"
        "Teams Used WinPhone", "Teams Used iOS", "Teams Used Android"
    ],
    "Office": [
        "Office Windows", "Office Mac", "Office Mobile", "Office Web"
        "Office Outlook", "Office Word", "Office Excel", "Office Powerpoint", "Office Onenote", "Office Teams"
        "Office Outlook Windows", "Office Word Windows", "Office Excel Windows", "Office Powerpoint Windows", "Office Onenote Windows", "Office Teams Windows"
        "Office Outlook Mac", "Office Word Mac", "Office Excel Mac", "Office Powerpoint Mac", "Office Onenote Mac", "Office Teams Mac"
        "Office Outlook Mobile", "Office Word Mobile", "Office Excel Mobile", "Office Powerpoint Mobile", "Office Onenote Mobile", "Office Teams Mobile"
        "Office Outlook Web", "Office Word Web", "Office Excel Web", "Office Powerpoint Web", "Office Onenote Web", "Office Teams Web"
    ],
    "Yammer": [
        "Yammer Used Web", "Yammer Used Mobile", "Yammer Used Others"
        "Yammer Used WinPhone", "Yammer Used Android", "Yammer Used iPad", "Yammer Used iPhone"
    ]
}

usage_extra = {
    "Teams": [
        "Yammer Platform Count"
    ]
}